<a href="https://colab.research.google.com/github/tomalkaziofficial-del/Malnutrition-Prediction/blob/main/NutriPredict_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import math
import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, ConfusionMatrixDisplay, cohen_kappa_score, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from boruta import BorutaPy

matplotlib.use("Agg")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False


# ==========================================
# 1. Configuration and Directories
# ==========================================
RAW_FILE = "BDKR81FL.DTA"
RANDOM_STATE = 42
OUTPUT_DIR = "outputs_clean_standard_ml"
SAVE_CONFUSION_MATRICES = True
TEST_SIZE_RATIO = 0.20  # 80:20 Split

DIRS = {
    "results": os.path.join(OUTPUT_DIR, "target_results"),
    "features": os.path.join(OUTPUT_DIR, "selected_features"),
    "reports": os.path.join(OUTPUT_DIR, "classification_reports"),
    "shap": os.path.join(OUTPUT_DIR, "shap"),
    "summary": os.path.join(OUTPUT_DIR, "final_summary")
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
for _dir in DIRS.values():
    os.makedirs(_dir, exist_ok=True)


# ==========================================
# 2. Features Mapping & Definitions
# ==========================================
FEATURE_MAP = {
    "age": "v012", "mother_education": "v106", "bmi": "v445", "mother_occupation": "v717",
    "mother_work_status": "v714", "marital_status": "v501", "father_education": "v701",
    "father_occupation": "v705", "wealth": "v190", "region": "v024", "residence": "v025",
    "household_members": "v136", "floor_material": "v127", "toilet": "v116", "water_source": "v113",
    "electricity": "v119", "television": "v121", "radio": "v120", "reading_newspaper": "v157",
    "listening_radio": "v158", "religion": "v130", "unmet_family_planning": "v624",
    "contraceptive_use": "v364", "desire_more_children": "v605", "child_age": "b19",
    "birth_order": "bord", "birth_interval": "b11", "birth_weight_reported": "m19",
    "child_size_birth": "m18", "total_children": "v201", "age_first_birth": "v212",
    "u5_children": "v137", "place_delivery": "m15", "antenatal_care": "m14",
    "postnatal_care": "m70", "breastfed_first_hour": "m34", "breastfeeding_duration": "m5",
    "diarrhea": "h11", "fever": "h22", "child_weight_raw": "hw2", "child_height_raw": "hw3",
}

CONTINUOUS_COLS_BASE = [
    "age", "bmi", "household_members", "total_children", "birth_interval", "age_first_birth",
    "child_age", "breastfeeding_duration", "birth_weight_reported", "antenatal_care",
    "child_bmi", "linear_growth_index", "child_dependency_ratio"
]


# ==========================================
# 3. Helper Functions
# ==========================================
def unique_keep_order(items):
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out

def get_preprocessed_feature_names(preprocessor):
    try:
        return preprocessor.get_feature_names_out()
    except Exception:
        feature_names = []
        for name, transformer, columns in preprocessor.transformers_:
            if name == "remainder": continue
            if hasattr(transformer, "named_steps"):
                last_step = list(transformer.named_steps.values())[-1]
                if hasattr(last_step, "get_feature_names_out"):
                    try: names = last_step.get_feature_names_out(columns)
                    except Exception: names = columns
                else: names = columns
            else:
                names = columns
            feature_names.extend(list(names))
        return feature_names

def remove_duplicate_cols(df):
    return df.loc[:, ~df.columns.duplicated()].copy()


# ==========================================
# 4. Data Preprocessing & Cleaning Methods
# ==========================================
def apply_iqr_outlier_removal(df, columns):
    df_cleaned = df.copy()
    for col in columns:
        if col in df_cleaned.columns:
            Q1 = df_cleaned[col].quantile(0.25)
            Q3 = df_cleaned[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            df_cleaned.loc[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound), col] = np.nan
    return df_cleaned

def load_and_preprocess_raw_data():
    raw = pd.read_stata(RAW_FILE, convert_categoricals=False)

    available_features = {k: v for k, v in FEATURE_MAP.items() if v in raw.columns}
    selected_raw_cols = list(dict.fromkeys(available_features.values()))

    raw_to_feature = {}
    used_raw_cols = set()
    for feature_name, raw_name in available_features.items():
        if raw_name not in used_raw_cols:
            raw_to_feature[raw_name] = feature_name
            used_raw_cols.add(raw_name)

    df = raw[selected_raw_cols].rename(columns=raw_to_feature)
    df = remove_duplicate_cols(df)

    df = df.replace([97, 98, 99, 999, 9999, 99999], np.nan)

    if "bmi" in df.columns:
        df["bmi"] = df["bmi"] / 100

    if "child_weight_raw" in df.columns and "child_height_raw" in df.columns:
        weight_kg = df["child_weight_raw"] / 10.0
        height_cm = df["child_height_raw"] / 10.0
        height_m = height_cm / 100.0

        df["child_bmi"] = weight_kg / (height_m ** 2)
        df.loc[(df["child_bmi"] < 10) | (df["child_bmi"] > 50), "child_bmi"] = np.nan

        if "child_age" in df.columns:
            df["linear_growth_index"] = height_cm / (df["child_age"] + 1)
        df = df.drop(columns=["child_weight_raw", "child_height_raw"])

    if "age_first_birth" in df.columns:
        df["teenage_mother_risk"] = (df["age_first_birth"] < 18).astype(int)
        df.loc[df["age_first_birth"].isna(), "teenage_mother_risk"] = np.nan

    if "u5_children" in df.columns and "household_members" in df.columns:
        df["child_dependency_ratio"] = np.where(df["household_members"] > 0, df["u5_children"] / df["household_members"], np.nan)

    if "birth_interval" in df.columns:
        df["short_birth_interval_risk"] = (df["birth_interval"] < 24).astype(int)
        df.loc[df["birth_interval"].isna(), "short_birth_interval_risk"] = np.nan

    if "child_size_birth" in df.columns:
        df["small_birth_size_flag"] = (df["child_size_birth"] >= 4).astype(int)
        df.loc[df["child_size_birth"].isna(), "small_birth_size_flag"] = np.nan

    df["stunting_z"] = raw["hw70"] / 100
    df["underweight_z"] = raw["hw71"] / 100
    df["wasting_z"] = raw["hw72"] / 100

    df[["stunting_z", "wasting_z", "underweight_z"]] = df[["stunting_z", "wasting_z", "underweight_z"]].replace([99.96, 99.97, 99.98, 99.99], np.nan)

    df.loc[(df["stunting_z"] < -6) | (df["stunting_z"] > 6), "stunting_z"] = np.nan
    df.loc[(df["wasting_z"] < -5) | (df["wasting_z"] > 5), "wasting_z"] = np.nan
    df.loc[(df["underweight_z"] < -6) | (df["underweight_z"] > 5), "underweight_z"] = np.nan

    df["stunting"] = (df["stunting_z"] < -2).astype(int)
    df["wasting"] = (df["wasting_z"] < -2).astype(int)
    df["underweight"] = (df["underweight_z"] < -2).astype(int)

    df = apply_iqr_outlier_removal(df, CONTINUOUS_COLS_BASE)

    return df

class SimpleStatsImputer:
    def __init__(self):
        self.cat_cols = []
        self.num_cols = []
        self.num_fill_values = {}
        self.cat_fill_values = {}
        self.cat_bounds = {}

    def fit(self, X):
        X = remove_duplicate_cols(X)
        temp = X.copy()
        for col in temp.columns:
            temp[col] = pd.to_numeric(temp[col], errors="coerce")
            valid_values = temp[col].dropna()

            if col in CONTINUOUS_COLS_BASE:
                self.num_cols.append(col)
                self.num_fill_values[col] = valid_values.mean() if len(valid_values) > 0 else 0
            else:
                self.cat_cols.append(col)
                if len(valid_values) > 0:
                    self.cat_fill_values[col] = valid_values.mode().iloc[0]
                    self.cat_bounds[col] = (valid_values.min(), valid_values.max())
                else:
                    self.cat_fill_values[col] = 0
        return self

    def transform(self, X):
        X = remove_duplicate_cols(X)
        temp = X.copy()
        for col in temp.columns:
            temp[col] = pd.to_numeric(temp[col], errors="coerce")

        for col in self.num_cols:
            if col in temp.columns:
                temp[col] = temp[col].fillna(self.num_fill_values.get(col, temp[col].median()))

        for col in self.cat_cols:
            if col in temp.columns:
                temp[col] = temp[col].fillna(self.cat_fill_values.get(col, 0))
                if col in self.cat_bounds:
                    lower, upper = self.cat_bounds[col]
                    temp[col] = temp[col].clip(lower=lower, upper=upper)
                temp[col] = np.round(temp[col]).astype(int).astype("category")

        return temp

def save_descriptive_statistics(df, output_dir):
    print("\nGenerating Descriptive Statistics...")
    num_stats = df.describe().T
    num_stats.to_csv(os.path.join(output_dir, "descriptive_stats_numerical.csv"))

    missing_stats = df.isnull().sum().to_frame(name='Missing_Values')
    missing_stats['Missing_Percentage'] = (missing_stats['Missing_Values'] / len(df)) * 100
    missing_stats.to_csv(os.path.join(output_dir, "descriptive_stats_missing.csv"))
    print("Descriptive statistics saved successfully.")

def build_preprocessor(X):
    X = remove_duplicate_cols(X)
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([("scaler", StandardScaler())])
    categorical_pipe = Pipeline([("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])

    transformers = [("num", numeric_pipe, numeric_cols)]
    if categorical_cols:
        transformers.append(("cat", categorical_pipe, categorical_cols))

    return ColumnTransformer(transformers, verbose_feature_names_out=False)


# ==========================================
# 5. Feature Selection (Boruta + Synchronized Plots)
# ==========================================
def generate_consistent_boruta_plots(boruta_df, X_arr, y_arr, feature_names, target_name, n_iterations=100):
    rng = np.random.RandomState(RANDOM_STATE)
    n_samples, n_features = X_arr.shape

    importance_records = {name: [] for name in feature_names}
    shadow_names = ["shadowMin", "shadowMean", "shadowMax"]
    for name in shadow_names:
        importance_records[name] = []

    for it in range(n_iterations):
        shadow_arr = X_arr.copy()
        for col_idx in range(n_features):
            shadow_arr[:, col_idx] = rng.permutation(shadow_arr[:, col_idx])

        X_with_shadow = np.hstack([X_arr, shadow_arr])

        rf_iter = RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_leaf=5, max_features="sqrt",
            class_weight="balanced_subsample", bootstrap=True,
            random_state=rng.randint(0, 1_000_000), n_jobs=-1
        )
        rf_iter.fit(X_with_shadow, y_arr)
        importances = rf_iter.feature_importances_

        real_importances = importances[:n_features]
        shadow_importances = importances[n_features:]

        for i, name in enumerate(feature_names):
            importance_records[name].append(real_importances[i])

        importance_records["shadowMin"].append(shadow_importances.min())
        importance_records["shadowMean"].append(shadow_importances.mean())
        importance_records["shadowMax"].append(shadow_importances.max())

    medians = {name: np.median(importance_records[name]) for name in importance_records}
    status_lookup = dict(zip(boruta_df["Feature"], boruta_df["Boruta_Status"]))
    for name in shadow_names:
        status_lookup[name] = "Shadow"

    status_priority = {"Confirmed": 0, "Tentative": 1, "Rejected": 2}

    sorted_features = sorted(
        feature_names,
        key=lambda f: (status_priority.get(status_lookup.get(f, "Rejected"), 2), -medians[f])
    )

    real_feature_order = sorted_features[::-1]
    ordered_all_names = shadow_names + real_feature_order
    box_colors = {"Confirmed": "limegreen", "Tentative": "gold", "Rejected": "tomato", "Shadow": "royalblue"}

    box_data = [importance_records[name] for name in ordered_all_names]

    fig, ax = plt.subplots(figsize=(18, 9))
    bp = ax.boxplot(box_data, labels=ordered_all_names, patch_artist=True, showfliers=True, widths=0.6)

    ax.set_yscale('symlog', linthresh=0.001)

    for patch, name in zip(bp["boxes"], ordered_all_names):
        status = status_lookup.get(name, "Rejected")
        patch.set_facecolor(box_colors.get(status, "lightgray"))
        patch.set_edgecolor("black")

    for median_line in bp["medians"]:
        median_line.set_color("black")

    ax.set_title(f"Boruta Feature Selection (Boxplot) for {target_name.capitalize()}", fontsize=16, fontweight="bold")
    ax.set_ylabel("Importance (Log Scale)", fontsize=12)
    # Boxplot x-axis labels made bold here
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10, fontweight="bold")

    legend_handles = [
        mpatches.Patch(color="royalblue", label="Shadow"),
        mpatches.Patch(color="tomato", label="Rejected"),
        mpatches.Patch(color="gold", label="Tentative"),
        mpatches.Patch(color="limegreen", label="Confirmed"),
    ]
    ax.legend(handles=legend_handles, loc="upper left", fontsize=10)
    plt.tight_layout()

    boxplot_path = os.path.join(DIRS["features"], f"{target_name}_boruta_boxplot.png")
    plt.savefig(boxplot_path, dpi=300, bbox_inches="tight")
    plt.close()

    bar_colors = [box_colors[status_lookup.get(f, "Rejected")] for f in real_feature_order]
    bar_heights = [medians[f] for f in real_feature_order]

    plt.figure(figsize=(12, 10))
    plt.barh(real_feature_order, bar_heights, color=bar_colors, edgecolor='black')

    plt.title(f"Boruta Median Feature Importance for {target_name.capitalize()}", fontsize=14, fontweight='bold')
    plt.xlabel("Median Feature Importance (100 Iterations)", fontsize=12)
    plt.ylabel("Features", fontsize=12)

    bar_legend_handles = [
        mpatches.Patch(color='limegreen', label='Confirmed'),
        mpatches.Patch(color='gold', label='Tentative'),
        mpatches.Patch(color='tomato', label='Rejected')
    ]
    plt.legend(handles=bar_legend_handles, loc='lower right')

    plt.tight_layout()
    barplot_path = os.path.join(DIRS["features"], f"{target_name}_boruta_plot.png")
    plt.savefig(barplot_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[*] Synchronized & Formatted Boruta Plots saved for {target_name}")


def run_boruta_selection(X_train, y_train, target_name):
    X_enc = X_train.copy()
    ohe_cols = [c for c in X_enc.columns if hasattr(X_enc[c], "cat") or X_enc[c].dtype == object]
    num_cols = [c for c in X_enc.columns if c not in ohe_cols]

    for col in num_cols: X_enc[col] = pd.to_numeric(X_enc[col], errors="coerce")
    if ohe_cols:
        oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_enc[ohe_cols] = oe.fit_transform(X_enc[ohe_cols].astype(str))
    for col in X_enc.columns: X_enc[col] = pd.to_numeric(X_enc[col], errors="coerce")

    X_arr = X_enc.values.astype(float)
    y_arr = y_train.values.astype(int)

    rf_boruta = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_leaf=5, max_features="sqrt", class_weight="balanced_subsample", bootstrap=True, random_state=RANDOM_STATE, n_jobs=-1)
    boruta = BorutaPy(estimator=rf_boruta, n_estimators="auto", perc=40, alpha=0.40, max_iter=150, random_state=RANDOM_STATE, verbose=0)

    print(f"\n[Boruta] Running for target: {target_name.upper()} ...")
    boruta.fit(X_arr, y_arr)
    print(f"[Boruta] Done for target: {target_name.upper()}")

    feature_names = list(X_train.columns)
    boruta_df = pd.DataFrame({
        "Feature": feature_names,
        "Boruta_Support": boruta.support_, "Boruta_Weak": boruta.support_weak_, "Boruta_Ranking": boruta.ranking_,
    })
    boruta_df["Boruta_Status"] = np.where(boruta_df["Boruta_Support"], "Confirmed", np.where(boruta_df["Boruta_Weak"], "Tentative", "Rejected"))

    generate_consistent_boruta_plots(boruta_df, X_arr, y_arr, feature_names, target_name, n_iterations=100)

    boruta_df = boruta_df.sort_values("Boruta_Ranking").reset_index(drop=True)
    boruta_df.to_csv(os.path.join(DIRS["features"], f"{target_name}_boruta_feature_status.csv"), index=False)

    confirmed = boruta_df.loc[boruta_df["Boruta_Support"], "Feature"].tolist()
    tentative = boruta_df.loc[boruta_df["Boruta_Weak"], "Feature"].tolist()

    selected = unique_keep_order(confirmed + tentative)
    if len(selected) < 8:
        top_ranked = boruta_df["Feature"].tolist()
        for f in top_ranked:
            if f not in selected: selected.append(f)
            if len(selected) >= 8: break
    return selected


# ==========================================
# 6. ML Models & Evaluation
# ==========================================
def get_models(pos_weight):
    models = {}

    lr_params = {'C': [0.1, 1, 10]}
    models["LR"] = RandomizedSearchCV(LogisticRegression(max_iter=5000, class_weight="balanced", solver="liblinear"),
                                      lr_params, n_iter=3, cv=3, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)

    cart_params = {'max_depth': [3, 5, 8], 'min_samples_leaf': [5, 10]}
    models["CART"] = RandomizedSearchCV(DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
                                        cart_params, n_iter=3, cv=3, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)

    rf_base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    rf_params = {"n_estimators": [300, 500], "max_depth": [10, 15], "min_samples_split": [5]}
    models["RF"] = RandomizedSearchCV(rf_base, rf_params, n_iter=3, cv=3, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)

    xgb_base = XGBClassifier(scale_pos_weight=min(pos_weight, 5.0), objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    xgb_params = {"n_estimators": [300, 500], "max_depth": [3, 5], "learning_rate": [0.05]}
    models["XGBM"] = RandomizedSearchCV(xgb_base, xgb_params, n_iter=3, cv=3, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)

    lgbm_base = LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    lgbm_params = {"n_estimators": [300, 500], "learning_rate": [0.05], "num_leaves": [31]}
    models["LGBM"] = RandomizedSearchCV(lgbm_base, lgbm_params, n_iter=3, cv=3, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)

    cat_base = CatBoostClassifier(auto_class_weights="Balanced", verbose=0, random_state=RANDOM_STATE, thread_count=-1, allow_writing_files=False)
    cat_params = {"iterations": [300, 500], "learning_rate": [0.05]}
    models["CAT"] = RandomizedSearchCV(cat_base, cat_params, n_iter=3, cv=3, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)

    return models

def get_safe_smote_ratio(y_train):
    pos = y_train.sum()
    neg = len(y_train) - pos
    if pos == 0: return 1.0
    return min(1.0, (neg / pos) * 0.75)

def strict_10fold_cv(pipe, X, y):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"accuracy": "accuracy", "f1": "f1", "precision": "precision", "recall": "recall", "roc_auc": "roc_auc"}

    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)

    return {
        "CV Accuracy": np.mean(scores["test_accuracy"]),
        "CV Precision": np.mean(scores["test_precision"]),
        "CV Sensitivity": np.mean(scores["test_recall"]),
        "CV Binary F1": np.mean(scores["test_f1"]),
        "CV ROC AUC": np.mean(scores["test_roc_auc"])
    }

def evaluate_metrics(y_true, y_prob, threshold=0.50):
    pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    acc = accuracy_score(y_true, pred)
    n = len(y_true)

    z_score = 1.96
    margin_error = z_score * math.sqrt((acc * (1 - acc)) / n) if n > 0 else 0
    ci_lower = max(0.0, acc - margin_error)
    ci_upper = min(1.0, acc + margin_error)
    ci_str = f"({ci_lower:.3f}, {ci_upper:.3f})"

    kappa = cohen_kappa_score(y_true, pred)
    binary_f1 = f1_score(y_true, pred, zero_division=0)
    weighted_f1 = f1_score(y_true, pred, average="weighted", zero_division=0)

    try: roc_auc = roc_auc_score(y_true, y_prob)
    except Exception: roc_auc = np.nan

    return {
        "Accuracy": acc, "95% CI": ci_str, "Cohen's kappa": kappa,
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Sensitivity": recall_score(y_true, pred, zero_division=0),
        "Specificity": specificity, "Binary F1": binary_f1,
        "Weighted F1": weighted_f1, "ROC AUC": roc_auc
    }

def save_classification_report(y_true, y_prob, threshold, target_name):
    pred = (y_prob >= threshold).astype(int)
    report_text = classification_report(y_true, pred, target_names=["Normal", "Malnourished"], digits=4, zero_division=0)
    report_text = report_text.replace("recall", "sensitivity")
    with open(os.path.join(DIRS["reports"], f"{target_name}_classification_report.txt"), "w") as f:
        f.write(report_text)
    print(f"\n--- {target_name.upper()} Classification Report ---")
    print(report_text)

def print_target_summary(eval_result, target_name, model_name):
    print(f"\n{target_name.upper()} SUMMARY (Best Model: {model_name}) ")
    print(f"Accuracy      : {eval_result['Accuracy']:.4f}")
    print(f"Precision     : {eval_result['Precision']:.4f}")
    print(f"Sensitivity   : {eval_result['Sensitivity']:.4f}")
    print(f"Specificity   : {eval_result['Specificity']:.4f}")
    print(f"Weighted F1   : {eval_result['Weighted F1']:.4f}")
    print(f"ROC AUC       : {eval_result['ROC AUC']:.4f}")
    print("=" * 60)

def save_confusion_matrix_image(y_true, y_prob, threshold, target_name, model_name):
    if not SAVE_CONFUSION_MATRICES: return
    pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Malnourished"])
    disp.plot(values_format="d")
    plt.title(f"{target_name.upper()} - {model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(DIRS["shap"], f"{target_name}_confusion_matrix.png"), dpi=300, bbox_inches="tight")
    plt.close()

def save_roc_auc_curve(y_true, model_probabilities, target_name, best_model_name=None):
    color_cycle = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
        "#9467bd", "#8c564b", "#17becf", "#e377c2"
    ]

    plt.figure(figsize=(8, 7))

    for idx, (model_name, y_prob) in enumerate(model_probabilities.items()):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        color = color_cycle[idx % len(color_cycle)]

        linestyle = "-"
        linewidth = 2.0
        if best_model_name is not None and model_name == best_model_name:
            linewidth = 3.2

        plt.plot(
            fpr, tpr, color=color, linestyle=linestyle, linewidth=linewidth,
            label=f"{model_name} (AUC = {auc_val:.3f})"
        )

    plt.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1.5, label="Chance (AUC = 0.500)")

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
    plt.ylabel("True Positive Rate (Sensitivity)", fontsize=12)
    plt.title(f"ROC-AUC Curve Comparison: {target_name.capitalize()}", fontsize=14, fontweight="bold")
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # ROC AUC Curve saved precisely as SVG format here
    plot_path = os.path.join(DIRS["shap"], f"{target_name}_roc_auc_curve.svg")
    plt.savefig(plot_path, format="svg", bbox_inches="tight")
    plt.close()
    print(f"[*] ROC-AUC curve plot saved as SVG to {plot_path}")

def _extract_fitted_estimator(model):
    if hasattr(model, "best_estimator_"):
        return model.best_estimator_
    return model

def save_shap(model, X, target_name, best_model_name=None):
    if not SHAP_AVAILABLE:
        print(f"[!] SHAP skipped for {target_name}: shap package not available/importable in this environment.")
        return
    try:
        fitted_estimator = _extract_fitted_estimator(model)

        sample_size = min(500, len(X))
        X_sample = X.sample(n=sample_size, random_state=RANDOM_STATE)

        # UPDATED SECTION: Safe handling for Tree vs Linear Explainers
        if best_model_name in ["RF", "XGBM", "LGBM", "CAT", "CART"]:
            explainer = shap.TreeExplainer(fitted_estimator)
            shap_values = explainer.shap_values(X_sample)

            # TreeExplainer on binary classification often returns a list of arrays (class 0, class 1)
            if isinstance(shap_values, list) and len(shap_values) == 2:
                shap_values = shap_values[1]
        else:
            background_size = min(100, len(X_sample))
            X_background = X_sample.sample(n=background_size, random_state=RANDOM_STATE)
            explainer = shap.LinearExplainer(fitted_estimator, X_background)
            shap_values = explainer.shap_values(X_sample)

        plt.figure()
        shap.summary_plot(shap_values, X_sample, show=False, max_display=10)
        plt.tight_layout()
        plt.savefig(os.path.join(DIRS["shap"], f"{target_name}_shap_summary.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[*] Explainable AI (SHAP, model: {best_model_name}) generated for {target_name}")
    except Exception as e:
        print(f"[!] SHAP Error for {target_name}: {e}")


# ==========================================
# 7. Main Training Pipeline
# ==========================================
def train_target(df, target_name):
    print("\n" + "="*60)
    print(f" PIPELINE INITIATED FOR TARGET: {target_name.upper()} ")
    print("="*60)

    classification_threshold = 0.50

    drop_cols = ["stunting", "wasting", "underweight", "stunting_z", "wasting_z", "underweight_z"]
    keep = [c for c in df.columns if c not in drop_cols]

    X = df[keep].copy()
    X = remove_duplicate_cols(X)
    y = df[target_name]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE_RATIO, stratify=y, random_state=RANDOM_STATE)

    missing_percent = X_train.isna().mean() * 100
    keep_cols = missing_percent[missing_percent < 40].index.tolist()
    X_train = X_train[keep_cols]
    X_test = X_test[keep_cols]

    stats_imp = SimpleStatsImputer()
    stats_imp.fit(X_train)
    X_train = stats_imp.transform(X_train)
    X_test = stats_imp.transform(X_test)

    X_combined = pd.concat([X_train, X_test])
    y_combined = pd.concat([y_train, y_test])
    processed_df = X_combined.copy()
    processed_df[target_name] = y_combined

    csv_filename = os.path.join(OUTPUT_DIR, f"{target_name}_preprocessed_clean_dataset.csv")
    processed_df.to_csv(csv_filename, index=False)
    print(f"[*] Cleaned data exported successfully.")

    default_features = [
        "child_bmi", "linear_growth_index", "teenage_mother_risk", "child_dependency_ratio",
        "short_birth_interval_risk", "small_birth_size_flag", "child_age", "child_size_birth",
        "birth_order", "birth_interval", "age", "mother_education", "father_education",
        "father_occupation", "mother_occupation", "mother_work_status", "bmi", "household_members",
        "wealth", "region", "residence", "toilet", "water_source", "electricity", "television",
        "radio", "listening_radio", "reading_newspaper", "religion", "marital_status",
        "breastfed_first_hour", "total_children", "u5_children", "place_delivery", "antenatal_care",
        "postnatal_care", "unmet_family_planning", "contraceptive_use", "desire_more_children", "diarrhea"
    ]

    candidate_features = [f for f in default_features if f in X_train.columns]
    candidate_features = unique_keep_order(candidate_features)
    X_train_boruta = X_train[candidate_features]

    selected = run_boruta_selection(X_train_boruta, y_train, target_name)
    selected = unique_keep_order(selected)

    X_train = X_train[selected]
    X_test = X_test[selected]

    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    models = get_models(pos_weight)

    rows = []
    best_model, best_model_name, best_test_prob, best_eval_result = None, None, None, None
    best_score = -999

    model_probabilities = {}

    for name, model in models.items():
        smote_ratio = get_safe_smote_ratio(y_train)

        sampler = SMOTE(sampling_strategy=smote_ratio, random_state=RANDOM_STATE, k_neighbors=7)
        pipe = ImbPipeline([("preprocess", build_preprocessor(X_train)), ("smote", sampler), ("model", model)])

        cv_result = strict_10fold_cv(pipe, X_train, y_train)
        pipe.fit(X_train, y_train)

        test_prob_raw = pipe.predict_proba(X_test)[:, 1]
        eval_result = evaluate_metrics(y_test, test_prob_raw, threshold=classification_threshold)

        score = np.mean([
            eval_result["Accuracy"], eval_result["Precision"], eval_result["Sensitivity"],
            eval_result["Specificity"], eval_result["Weighted F1"], eval_result["Binary F1"],
            eval_result["ROC AUC"]
        ])

        model_probabilities[name] = test_prob_raw

        row = {
            "Model": name, "Threshold": classification_threshold,
            "Accuracy": eval_result["Accuracy"], "95% CI": eval_result["95% CI"],
            "Cohen's kappa": eval_result["Cohen's kappa"], "Precision": eval_result["Precision"],
            "Sensitivity": eval_result["Sensitivity"], "Specificity": eval_result["Specificity"],
            "Weighted F1": eval_result["Weighted F1"], "Binary F1": eval_result["Binary F1"],
            "ROC AUC": eval_result["ROC AUC"], **cv_result, "Selection Score": score
        }
        rows.append(row)

        if score > best_score:
            best_score = score
            best_model = pipe
            best_model_name = name
            best_test_prob = test_prob_raw
            best_eval_result = eval_result

        print(f"[*] Model {name} evaluation complete.")

    result_df = pd.DataFrame(rows)
    result_df = result_df.sort_values(by=["Selection Score"], ascending=False)
    result_df.to_csv(os.path.join(DIRS["results"], f"{target_name}_results.csv"), index=False)

    save_classification_report(y_test, best_test_prob, classification_threshold, target_name)
    save_confusion_matrix_image(y_test, best_test_prob, classification_threshold, target_name, best_model_name)
    print_target_summary(best_eval_result, target_name, best_model_name)

    save_roc_auc_curve(y_test, model_probabilities, target_name, best_model_name=best_model_name)

    try:
        model_inner = best_model.named_steps["model"]
        preprocessor = best_model.named_steps["preprocess"]
        transformed_data = preprocessor.transform(X_train)
        feature_names = get_preprocessed_feature_names(preprocessor)
        if len(feature_names) != transformed_data.shape[1]: feature_names = [f"Feature_{i}" for i in range(transformed_data.shape[1])]
        transformed = pd.DataFrame(transformed_data, columns=list(feature_names), index=X_train.index)
        save_shap(model_inner, transformed, target_name, best_model_name=best_model_name)
    except Exception as e:
        print(f"[!] Error preparing SHAP for {target_name}: {e}")

    return result_df


# ==========================================
# 8. Execution Function
# ==========================================
def main():
    print("\n" + "="*60)
    print(" INITIATING METHODOLOGY PIPELINE ")
    print("="*60)

    df = load_and_preprocess_raw_data()
    save_descriptive_statistics(df, DIRS["summary"])

    if "child_age" in df.columns:
        stunting_source_df = df[(df["child_age"].astype(float) >= 6) & (df["child_age"].astype(float) <= 59)].copy()
    else:
        stunting_source_df = df.copy()

    stunting_df = stunting_source_df[stunting_source_df["stunting_z"].notna()].copy()
    wasting_df = df[df["wasting_z"].notna()].copy()
    underweight_df = df[df["underweight_z"].notna()].copy()

    stunting_results = train_target(stunting_df, "stunting")
    wasting_results = train_target(wasting_df, "wasting")
    underweight_results = train_target(underweight_df, "underweight")

    all_results = pd.concat([
        stunting_results.assign(Target="stunting"),
        wasting_results.assign(Target="wasting"),
        underweight_results.assign(Target="underweight")
    ], ignore_index=True)

    all_results.to_csv(os.path.join(DIRS["summary"], "all_target_results.csv"), index=False)

    print("\n" + "="*60)
    print(" PIPELINE EXECUTION FINISHED SUCCESSFULLY ")
    print("="*60)

if __name__ == "__main__":
    main()


 INITIATING METHODOLOGY PIPELINE 

Generating Descriptive Statistics...
Descriptive statistics saved successfully.

 PIPELINE INITIATED FOR TARGET: STUNTING 
[*] Cleaned data exported successfully.

[Boruta] Running for target: STUNTING ...
[Boruta] Done for target: STUNTING


/tmp/ipykernel_2142/880371987.py:329: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(box_data, labels=ordered_all_names, patch_artist=True, showfliers=True, widths=0.6)


[*] Synchronized & Formatted Boruta Plots saved for stunting
[*] Model LR evaluation complete.
[*] Model CART evaluation complete.
[*] Model RF evaluation complete.
[*] Model XGBM evaluation complete.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=3. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[*] Model LGBM evaluation complete.


...

[*] Model LGBM evaluation complete.


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=3. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[*] Model CAT evaluation complete.

--- UNDERWEIGHT Classification Report ---
              precision    sensitivity  f1-score   support

      Normal     0.9037    0.9487    0.9256       643
Malnourished     0.7898    0.6561    0.7168       189

    accuracy                         0.8822       832
   macro avg     0.8468    0.8024    0.8212       832
weighted avg     0.8778    0.8822    0.8782       832


UNDERWEIGHT SUMMARY (Best Model: LGBM) 
Accuracy      : 0.8822
Precision     : 0.7898
Sensitivity   : 0.6561
Specificity   : 0.9487
Weighted F1   : 0.8782
ROC AUC       : 0.9155
[*] ROC-AUC curve plot saved as SVG to outputs_clean_standard_ml/shap/underweight_roc_auc_curve.svg


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[*] Explainable AI (SHAP, model: LGBM) generated for underweight

 PIPELINE EXECUTION FINISHED SUCCESSFULLY 


In [ ]:
pip install Boruta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 3.7 MB/s eta 0:00:00


In [ ]:
!pip install catboost xgboost lightgbm imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.1 MB/s eta 0:00:00
